In [2]:
import os, sys
import pandas as pd
import polars as pl
import pyarrow
import numpy as np
import json
import time
from datetime import date, datetime

In [3]:
data_2019 = "//depot.engr.oregonstate.edu/mime_u1/dalziel/Safe Graph Data/Weekly Patterns/2019_Weekly_Patterns"
example = os.listdir(data_2019)[2]

In [4]:
cols_to_read = ["safegraph_place_id",
                "location_name",
                "street_address",
                "city",
                "region",
                "postal_code",
                "iso_country_code",
                "date_range_start",
                "raw_visit_counts",
                "raw_visitor_counts",
                "visits_by_day",
                "visits_by_each_hour",
                "poi_cbg",
                "visitor_home_cbgs",
                "visitor_daytime_cbgs",
                "visitor_country_of_origin",
                "distance_from_home",
                "median_dwell",
                "bucketed_dwell_times"]
schema_overrides = {"date_range_start": pl.Datetime,
                    "distance_from_home": pl.Int64}

In [5]:
read = pl.scan_csv(os.path.join(data_2019, example), 
                   schema_overrides=schema_overrides).select(cols_to_read).with_columns([pl.col("visits_by_each_hour").str.json_decode(pl.List(pl.Int64)),
                                                                                         pl.col("visits_by_day").str.json_decode(pl.List(pl.Int64))])

In [6]:
cols_to_select = [
    "safegraph_place_id",
    "city",
    "region",
    "date_range_start",
    "visits_by_each_hour"
]

In [7]:
read = (
    read
    .filter(pl.col("iso_country_code") == "US",
            pl.col("city") == "Portland",
            pl.col("region") == "OR")
    .select(cols_to_select)
    .with_columns(t = (pl.int_ranges(0, pl.col("visits_by_each_hour").list.len())))
    .explode(["t", "visits_by_each_hour"])
    .rename({"visits_by_each_hour": "visits"})
)

In [16]:
read = read.collect()

ComputeError: out of memory

<b> Augment with FEMA data

In [9]:
import pandas as pd
from shapely.geometry import Point
import geopandas as gpd

In [10]:
target_folder = r"\\depot.engr.oregonstate.edu\mime_u1\dalziel\Safe Graph Data\Weekly Patterns\Digital_Twins_Analysis\temporary_stash_very_heavy"
core_places_data_2019 = "//depot.engr.oregonstate.edu/mime_u1/dalziel/Safe Graph Data/Core Places Data/CoreRecords-CORE_POI-2019_03-2020-03-25"
city = "Portland"

In [11]:
def find_pois_contained_in_fema_geometry(fema_data, safegraph_place_data):
    # given a fema dataset and a safegraph dataset, loop through the sg pois and see what fema build id it"s within, then write the poi id to fema dataset as match
    places_city = safegraph_place_data[safegraph_place_data["city"]==city]
    sg_match = ["" for _ in range(len(fema_data))]
    for poi in places_city.itertuples(index= False):
        poi_point = Point(getattr(poi, "longitude"), getattr(poi, "latitude"))
        poi_id = getattr(poi, "safegraph_place_id")
        fema_data["contains"] = fema_data["geometry"].contains(poi_point)
        match_indices = fema_data[fema_data["contains"]==True].index
        if match_indices.size == 0:
            continue
        match_ind = np.random.choice(match_indices)
        sg_match[match_ind] = poi_id
    fema_data["sg_match"] = sg_match

    return fema_data


In [12]:
gdf = gpd.read_file(r"\\stak.engr.oregonstate.edu\Users\len2\research\fema_data\raw\Portland_clip.gpkg")
gdf_subset = gdf[["BUILD_ID", "OCC_CLS", "PRIM_OCC", "SQMETERS", "SQFEET", "CENSUSCODE", "UUID", "geometry"]]
gdf_nonresidential = gdf_subset[gdf_subset["OCC_CLS"] != "Residential"]


places_data = pd.read_csv(r"\\depot.engr.oregonstate.edu\mime_u1\dalziel\Safe Graph Data\Weekly Patterns\Digital_Twins_Analysis\temporary_stash_very_heavy\2019_Portland.csv")
places_centroid = gpd.GeoDataFrame(
    places_data,
    geometry = gpd.points_from_xy(
        places_data.longitude,
        places_data.latitude
    ),
    crs = "EPSG:4326"
)


In [13]:
matches = gpd.sjoin(gdf_nonresidential, places_centroid, predicate="contains", how = "left")
matches

,BUILD_ID,OCC_CLS,PRIM_OCC,SQMETERS,SQFEET,CENSUSCODE,UUID,geometry,index_right,safegraph_place_id,...,naics_code,latitude,longitude,street_address,city,region,postal_code,iso_country_code,open_hours,category_tags
29,3682969,Unclassified,Unclassified,49.627411,534.184509,41035970300,{f47b2433-461c-48f9-9fa5-c557ea563568},"MULTIPOLYGON (((-121.87448 42.03853, -121.8745...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,1966916,Utility and Misc,Ground,95.266861,1025.442993,41035970300,{8e91b87c-ab4c-440f-a4f8-529456f7d0dd},"MULTIPOLYGON (((-121.97102 42.06568, -121.971 ...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,1966920,Utility and Misc,Ground,220.338730,2371.704102,41035970300,{0d9ab24e-30f2-4a75-9648-d5feb42d0986},"MULTIPOLYGON (((-121.97105 42.06782, -121.9711...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,1966922,Utility and Misc,Ground,140.506042,1512.392944,41035970300,{c5db3f30-2cea-4f86-a2a6-0fd2ef267063},"MULTIPOLYGON (((-121.97361 42.06833, -121.9734...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,3683134,Unclassified,Unclassified,173.898010,1871.820801,41035970300,{29b9830b-721e-4376-85b1-fc59225a713c},"MULTIPOLYGON (((-121.89341 42.03959, -121.8934...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
992553,3567454,Agriculture,Agriculture,292.398071,3147.343506,41067033400,{72d31d42-4da1-482a-a041-c9005c0da0a9},"MULTIPOLYGON (((-123.14557 45.6879, -123.14558...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
992554,3567479,Agriculture,Agriculture,114.079170,1227.936768,41067033400,{2f239e59-d412-4eaa-9cca-009e299cf543},"MULTIPOLYGON (((-123.14588 45.68814, -123.1457...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
992561,3568057,Unclassified,Unclassified,268.252899,2887.447266,41067033400,{5a51517d-2b08-4667-b9e2-3970be14e0c7},"MULTIPOLYGON (((-123.14387 45.70291, -123.1436...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
992576,3568283,Agriculture,Agriculture,469.387299,5052.437988,41067033400,{f3d61205-86e3-4585-b556-9b1fa40abafe},"MULTIPOLYGON (((-123.14247 45.71477, -123.1428...",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
read_pd = read.to_pandas()
matches["safegraph_place_id"] = matches["safegraph_place_id"].astype(str)
read_pd["safegraph_place_id"] = read_pd["safegraph_place_id"].astype(str)
matches.join(read_pd, on = "safegraph_place_id", how = "left")

ComputeError: out of memory